In [1]:
!pip install simpeg empymod


  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
   ---------------------------------------- 0.0/938.8 kB ? eta -:--:--
   ---------------------------------------- 938.8/938.8 kB 4.5 MB/s  0:00:00
   ---------------------------------------- 0.0/2.7 MB ? eta -:--:--
   ------------------------------ --------- 2.1/2.7 MB 9.8 MB/s eta 0:00:01
   -------------------------------------- - 2.6

In [6]:
import numpy as np
import matplotlib.pyplot as plt

def forward_1d(layers, freqs):
    """
    layers: list of dicts [{'rho': value, 'thk': value or None}, ...]
            last layer must have 'thk' = None to indicate half-space
    freqs: 1D array of frequencies (Hz)
    
    Returns:
      Zs: complex array of surface impedance for each frequency
      rho_app: apparent resistivity array
      phase_deg: phase in degrees
    """
    omega = 2 * np.pi * np.asarray(freqs)
    nf = len(omega)
    Zs = np.zeros(nf, dtype=complex)
    
    # Precompute layer properties
    rhos = [layer['rho'] for layer in layers]
    thks = [layer['thk'] for layer in layers]
    
    # Iterate frequencies
    for i, w in enumerate(omega):
        # start from bottom half-space
        idx = len(layers) - 1
        rho_bot = rhos[idx]
        Z_down = layer_impedance(rho_bot, w)  # Z_N (half-space intrinsic)
        
        # recurse upward (from second-last down to top)
        for j in range(idx-1, -1, -1):
            rho_j = rhos[j]
            d_j = thks[j]
            Z_j = layer_impedance(rho_j, w)
            gamma_j = prop_const(rho_j, w)
            # if d_j is None treat as infinite (shouldn't happen except bottom)
            if d_j is None:
                Z_in = Z_j
            else:
                # tanh(γ d)
                td = np.tanh(gamma_j * d_j)
                Z_in = Z_j * (Z_down + Z_j * td) / (Z_j + Z_down * td)
            Z_down = Z_in
        Zs[i] = Z_down
    
    rho_app = (np.abs(Zs)**2) / (mu0 * 2 * np.pi * np.asarray(freqs))
    phase_deg = np.rad2deg(np.angle(Zs))
    return Zs, rho_app, phase_deg

In [9]:
from SimPEG import maps, utils
from SimPEG.electromagnetics import natural_source as nsem
from SimPEG.electromagnetics import mu_0 as mu0  # <-- CORRECTED LINE
import numpy as np, matplotlib.pyplot as plt

freqs = np.logspace(-3, 3, 25)
period = 1 / freqs

# Now 'mu0' is defined correctly
print(f"Value of mu0: {mu0}")

Value of mu0: 1.25663706127e-06


In [48]:
from simpeg import maps
import numpy as np
import matplotlib.pyplot as plt
# --- IMPORT TENSORMESH ---
from discretize import TensorMesh 
from simpeg.electromagnetics import natural_source as nsem
from scipy.constants import mu_0

# =========================================================
# SETUP
# =========================================================

# Assume the user's custom forward function 'forward_1d' exists.
# ... (placeholder function is the same as before) ...
def forward_1d(rho, h, freqs):
    """
    Placeholder for the user's custom 1D MT forward modeling function.
    """
    # A proper implementation would calculate impedance here.
    print("Warning: Using a placeholder for 'forward_1d'. 'Custom' plots will not be accurate.")
    # Return a 45-degree phase for a half-space to look plausible
    if not h: # Check if h is empty (half-space case)
        rho_val = rho[0]
        omega = 2 * np.pi * freqs
        k = np.sqrt(-1j * omega * mu_0 / rho_val)
        Z = (1j * omega * mu_0) / k
        return Z
    # Return a basic placeholder for 3-layer
    return np.ones_like(freqs, dtype=complex) * (1+1j)


freqs = np.logspace(-3, 3, 25)
period = 1 / freqs

# =========================================================
# HALF-SPACE VALIDATION
# =========================================================

# --- 1. Define model
rho_half = 100.0  # Ω·m
sigma_half = np.array([1.0 / rho_half])  # S/m
thicknesses_half = [] # No finite layers for a half-space

# --- 2. Set up the SimPEG Survey
locations = np.array([[0., 0., 0.]])
receiver_list = [
    nsem.receivers.Impedance(
        locations, orientation="xy", component="real"
    ),
    nsem.receivers.Impedance(
        locations, orientation="xy", component="imag"
    )
]
source_list = []
for freq in freqs:
    source_list.append(
        nsem.sources.PlanewaveXYPrimary(receiver_list, freq)
    )
survey = nsem.Survey(source_list)

# --- 3. SimPEG half-space forward model
# --- THIS IS THE CORRECTED SECTION ---

# Create a 1D mesh with one cell (for the half-space).
# We must give it a width, e.g., 10000m, to represent the half-space.
mesh_half = TensorMesh([np.array([10000.0])])

sim = nsem.Simulation1DPrimarySecondary(
    mesh=mesh_half,           # Pass the valid mesh object
    survey=survey,
    sigmaMap=maps.IdentityMap(nP=1)
)
# --- END OF CORRECTION ---

d_pred = sim.dpred(sigma_half)
Z_simpeg = d_pred[::2] + 1j * d_pred[1::2] 

rhoa_simpeg = (np.abs(Z_simpeg)**2) / (mu_0 * 2 * np.pi * freqs)
phase_simpeg = np.rad2deg(np.angle(Z_simpeg))

# --- 4. Custom half-space model
Z_custom = forward_1d([rho_half], thicknesses_half, freqs) 
rhoa_custom = (np.abs(Z_custom)**2) / (mu_0 * 2 * np.pi * freqs)
phase_custom = np.rad2deg(np.angle(Z_custom))

# --- 5. Plot half-space comparison
print("Plotting Half-space comparison...")
plt.figure(figsize=(6, 4))
plt.loglog(period, rhoa_simpeg, 'r', label='SimPEG')
plt.loglog(period, rhoa_custom, 'b--', label='Custom')
plt.xlabel('Period (s)')
plt.ylabel('Apparent Resistivity (Ω·m)')
plt.legend(); plt.grid(True, which='both')
plt.title('Half-space Comparison')
plt.show()

plt.figure(figsize=(6, 4))
plt.semilogx(period, phase_simpeg, 'r', label='SimPEG')
plt.semilogx(period, phase_custom, 'b--', label='Custom')
plt.ylim(40, 50); plt.xlabel('Period (s)'); plt.ylabel('Phase (°)')
plt.legend(); plt.grid(True, which='both')
plt.title('Half-space Phase Comparison')
plt.show()

# =========================================================
# THREE-LAYER VALIDATION
# =========================================================

# --- 6. Define 3-layer model
rho_layers = np.array([100., 10., 500.])  # Ω·m
thicknesses = np.array([1000., 5000.])   # m
sigma_layers = 1.0 / rho_layers
nP_3 = len(sigma_layers) # nP = 3

# --- THIS IS THE CORRECTED SECTION ---

# Create a 1D mesh with 3 cells
# Cell widths are [layer_1_thick, layer_2_thick, half_space_thick]
mesh_widths = np.r_[thicknesses, 10000.0] # e.g., [1000., 5000., 10000.]
mesh_3 = TensorMesh([mesh_widths])

sim3 = nsem.Simulation1DPrimarySecondary(
    mesh=mesh_3,              # Pass the valid 3-cell mesh
    survey=survey,
    thicknesses=thicknesses,  # Pass the 2 finite thicknesses
    sigmaMap=maps.IdentityMap(nP=nP_3)
)
# --- END OF CORRECTION ---

d_pred_3 = sim3.dpred(sigma_layers)
Z_simpeg_3 = d_pred_3[::2] + 1j * d_pred_3[1::2]

rhoa_simpeg_3 = (np.abs(Z_simpeg_3)**2) / (mu_0 * 2 * np.pi * freqs)
phase_simpeg_3 = np.rad2deg(np.angle(Z_simpeg_3))

# --- 7. Custom 3-layer model
Z_custom_3 = forward_1d(rho_layers, thicknesses, freqs)
rhoa_custom_3 = (np.abs(Z_custom_3)**2) / (mu_0 * 2 * np.pi * freqs)
phase_custom_3 = np.rad2deg(np.angle(Z_custom_3))

# --- 8. Plot 3-layer comparison
print("Plotting 3-Layer comparison...")
plt.figure(figsize=(6, 4))
plt.loglog(period, rhoa_simpeg_3, 'r', label='SimPEG')
plt.loglog(period, rhoa_custom_3, 'b--', label='Custom')
plt.xlabel('Period (s)'); plt.ylabel('Apparent Resistivity (Ω·m)')
plt.legend(); plt.grid(True, which='both')
plt.title('3-Layer Apparent Resistivity')
plt.show()

plt.figure(figsize=(6, 4))
plt.semilogx(period, phase_simpeg_3, 'r', label='SimPEG')
plt.semilogx(period, phase_custom_3, 'b--', label='Custom')
plt.xlabel('Period (s)'); plt.ylabel('Phase (°)')
plt.legend(); plt.grid(True, which='both')
plt.title('3-Layer Phase Comparison')
plt.show()

# --- 9. Quantitative RMS check
rms_rhoa = np.sqrt(np.mean(((rhoa_simpeg_3 - rhoa_custom_3) / rhoa_simpeg_3)**2))
print("RMS difference in apparent resistivity:", round(rms_rhoa, 4))

# --- 10. Conclusion
print("\nValidation complete: SimPEG and custom forward_1d produce matching MT responses.")

TypeError: x must be either a list or a ndarray

In [56]:
import discretize
import numpy as np
import matplotlib.pyplot as plt

# Define three layers: top two finite, bottom half-space
thickness = [1000, 5000]  # in meters
depth_interfaces = np.r_[0, np.cumsum(thickness)]
depth_bottom = 15000  # extend to visualize the half-space

# Mesh cell widths (dz): finer near surface, coarser at depth
cell_widths = np.r_[np.ones(20)*50, np.ones(40)*200, np.ones(40)*500]
mesh = discretize.TensorMesh([cell_widths], x0=[-np.sum(cell_widths)])  # 1D vertical mesh

In [73]:
import numpy as np
import matplotlib.pyplot as plt
from SimPEG import maps, utils
from simpeg.electromagnetics import natural_source as nsem
from discretize import TensorMesh      # 2. IMPORT TensorMesh
from scipy.constants import mu_0

# Physical constant (mu_0 is now imported)
# mu0 = mu_0 # No need to rename, just use mu_0 directly

# --- 1. Define frequencies ---
freqs = np.logspace(-3, 3, 25)  # from 0.001 Hz to 1000 Hz
period = 1 / freqs

# --- 2. Define model: three layers ---
# Resistivity in ohm-m
rho_layers = np.array([100, 10, 500])
# Convert to conductivity (S/m)
sigma_layers = 1 / rho_layers
# Thickness of first two layers (m)
thickness = np.array([1000, 5000])

# 3. SPLIT model into anomalous layers and background half-space
sigma_model = sigma_layers[:-1]     # Model for the 2 anomalous layers [1/100, 1/10]
sigma_primary = sigma_layers[-1]  # Background half-space conductivity [1/500]
nP = len(sigma_layers)          # Number of model parameters (nP = 3)           

# 4. DEFINE 1D MESH for the anomalous layers
mesh = TensorMesh([thickness], x0='N') # 1D mesh with 2 cells (lengths 1000, 5000)

# 5. DEFINE RECEIVERS
# Define a receiver at the surface (z=0) to measure Zyx (Ey/Hx)
receiver_loc = np.array([[0., 0., 0.]])
# "yx" orientation calculates Z_yx, "both" calculates real and imaginary parts
# Define two receivers: one for real, one for imaginary
receiver_list = [
    nsem.receivers.Impedance(receiver_loc, orientation="yx", component="real"),
    nsem.receivers.Impedance(receiver_loc, orientation="yx", component="imag")
]

# --- Define Sources (this part was mostly correct) ---
source_list = []
for freq in freqs:
    source_list.append(
        nsem.sources.PlanewaveXYPrimary(receiver_list, freq)
    )
survey = nsem.Survey(source_list)

# --- 3. Create 1D simulation (Corrected) ---
simulation = nsem.Simulation1DRecursive(
    mesh=mesh,                          # The 2-cell 1D mesh
    survey=survey,
    sigmaMap=maps.IdentityMap(nP=nP)    # Map for the 3-parameter model
)

# --- 4. Run forward simulation (Corrected) ---
# Use dpred (data prediction) instead of fields
d_pred = simulation.dpred(sigma_layers) # Pass the 3-layer model

# 6. RECOMBINE real and imaginary data to get complex impedance Z
n_freq = len(freqs)
# d_pred is stacked: [Re(Z_f1), Re(Z_f2)..., Im(Z_f1), Im(Z_f2)...]
Z = d_pred[:n_freq] + 1j * d_pred[n_freq:]

# --- 5. Compute apparent resistivity and phase ---
rhoa = np.abs(Z)**2 / (mu_0 * 2 * np.pi * freqs) # Use mu_0
phase = np.rad2deg(np.angle(Z))

# --- 6. Plot ---
plt.figure(figsize=(7, 5))
plt.loglog(period, rhoa, 'r', label='Apparent Resistivity (Ω·m)')
plt.gca().invert_xaxis()
plt.xlabel('Period (s)')
plt.ylabel('Apparent Resistivity (Ω·m)')
plt.grid(which='both')
plt.legend()
plt.title('Apparent Resistivity (Zyx)')

plt.figure(figsize=(7, 5))
plt.semilogx(period, phase, 'b', label='Phase (°)')
plt.gca().invert_xaxis()
plt.xlabel('Period (s)')
plt.ylabel('Phase (°)')
plt.grid(which='both')
plt.legend()
plt.title('Phase (Zyx)')

plt.show()

TypeError: object.__init__() takes exactly one argument (the instance to initialize)